In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
docs = loader.load()

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(docs)

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [5]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(texts, embeddings)

In [6]:
vectorstore.save_local("./faiss_index")

In [7]:
vectorstore

In [8]:
vectorstore = None

In [9]:
vectorstore

In [10]:
vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [11]:
vectorstore

In [12]:
retriever = vectorstore.as_retriever()

In [13]:
query = "국가 기술 자격 중 기사 자격증을 취득하면 얼마를 받을 수 있을까?"

In [14]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트: {context}

질문: {question}
"""
)

In [15]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("google_genai:gemini-3.1-flash-lite")

In [17]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

In [18]:
response = chain.invoke(query)

In [19]:
response

'국가 기술 자격 중 기사 자격증을 취득할 경우, 50만 원의 축하금(1회성)을 받을 수 있으며, 매월 10만 원의 자격 수당을 받을 수 있습니다.'

In [20]:
query = "나는 부양 가족이 없는데 가족 수당은 얼마 받을 수 있을까?"
response = chain.invoke(query)
response

'제공된 컨텍스트에 따르면, 가족 수당은 부양 가족이 있는 자를 대상으로 지급됩니다. 부양 가족이 없는 경우에 대한 가족 수당 지급 내용은 명시되어 있지 않습니다.'

In [22]:
query = "나는 배우자가 있고, 자녀가 5명인데 가족 수당은 얼마 받을 수 있을까?"
response = chain.invoke(query)
response

'제공된 컨텍스트에 따르면, 가족 수당은 다음과 같이 지급됩니다.\n\n*   **배우자:** 5만 원\n*   **자녀:** 1인당 월 3만 원\n\n질문하신 경우, 배우자 수당 5만 원과 자녀 5명에 대한 수당(5명 × 3만 원 = 15만 원)을 합하여 **총 월 20만 원**의 가족 수당을 받을 수 있습니다. (단, 배우자 수당을 받기 위해서는 등본 제출이 필수입니다.)'